In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sabzenergy.climate import create_cutout
from sabzenergy.wind import calculate_wind_potential

plt.style.use('ggplot')


In [ ]:
SITES = {
    'Coastal_North': {'lat': 54.1, 'lon': 8.5},
    'Inland_North': {'lat': 53.0, 'lon': 9.0},
    'Central': {'lat': 51.0, 'lon': 10.0},
    'South_West': {'lat': 48.5, 'lon': 8.0},
    'South_East': {'lat': 48.5, 'lon': 12.0}
}
YEAR = 2023

# Dictionary to hold the generation timeseries for each site
site_generation = {}

for name, coords in SITES.items():
    print(f'Processing {name}...')
    # In a real scenario, you would create a cutout for each specific coordinate.
    # Here we simulate fetching the cutout for a 0.5 degree box around the point.
    # cutout = create_cutout(name, YEAR, output_dir='../../data', padding=0.5)
    # wind_gen = calculate_wind_potential(cutout, turbine='Vestas_V112_3MW').mean(['x', 'y']).to_series()
    # site_generation[name] = wind_gen
    pass


In [ ]:
# SYNTHETIC DATA GENERATION (Replace with actual 'site_generation' from atlite above)
np.random.seed(42)
time_index = pd.date_range(start='2023-01-01', end='2023-12-31 23:00', freq='H')

# Generating synthetic wind capacity factors using Weibull-like distributions
df = pd.DataFrame(index=time_index)
df['Coastal_North'] = np.clip(np.random.weibull(2.0, len(time_index)) * 0.4, 0, 1)
df['Inland_North'] = np.clip(np.random.weibull(1.8, len(time_index)) * 0.3, 0, 1)
df['Central'] = np.clip(np.random.weibull(1.7, len(time_index)) * 0.25, 0, 1)
df['South_West'] = np.clip(np.random.weibull(1.5, len(time_index)) * 0.2, 0, 1)
# South East is made to be slightly uncorrelated with Coastal North
df['South_East'] = np.clip(np.random.weibull(1.5, len(time_index)) * 0.2 + (0.1 - df['Coastal_North']*0.2), 0, 1)


In [ ]:
stats = pd.DataFrame({
    'Mean (Yield)': df.mean(),
    'Std Dev (Risk)': df.std()
})

plt.figure(figsize=(10, 6))
sns.scatterplot(data=stats, x='Std Dev (Risk)', y='Mean (Yield)', s=200, color='blue')

for i, row in stats.iterrows():
    plt.text(row['Std Dev (Risk)'] + 0.002, row['Mean (Yield)'], i, fontsize=12)

plt.title('Site Selection: Yield vs. Volatility')
plt.xlabel('Standard Deviation (Lower is better)')
plt.ylabel('Mean Capacity Factor (Higher is better)')
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Spatial Correlation of Wind Generation Between Sites')
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))

for column in df.columns:
    # Sort values descending
    sorted_values = np.sort(df[column])[::-1]
    # Calculate percentage of time
    percentile = np.arange(1, len(sorted_values) + 1) / len(sorted_values) * 100
    plt.plot(percentile, sorted_values, label=column)

plt.title('Wind Generation Duration Curve')
plt.xlabel('Percentage of Year (%)')
plt.ylabel('Capacity Factor')
plt.legend()
plt.show()
